# Generacion y subida de iabd05_sensores.json a S3

Este notebook genera un JSON de sensores y lo sube a S3.

In [14]:
import json
import os
from datetime import datetime, timedelta

import boto3
from faker import Faker
from dotenv import load_dotenv 

load_dotenv()

# 1) Constantes solicitadas para S3 y sesion boto3
AWS_REGION = "us-east-1"
S3_BUCKET = "s3-streamlit-ra2"
S3_KEY = "data/sensores/iabd05_sensores.json"

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")

# 2) Configuracion del fichero local
OUTPUT_FILE = "iabd05_sensores.json"
TARGET_SIZE_MB = 10
TARGET_SIZE_BYTES = TARGET_SIZE_MB * 1024 * 1024

In [ ]:
# Genera registros hasta superar
fake = Faker('es_ES')
Faker.seed(42)

lat_base = 43.370589
lon_base = -3.220316
start_ts = datetime(2025, 2, 1, 0, 0, 0)
estados = ['Activo', 'Inactivo', 'Mantenimiento']

records = []
i = 0

while True:
    record = {
        'sensor_id': f'sensor_{(i % 50) + 1:02d}',
        'ubicacion': {
            'latitud': round(lat_base + float(fake.pyfloat(min_value=-0.08, max_value=0.08, right_digits=6)), 6),
            'longitud': round(lon_base + float(fake.pyfloat(min_value=-0.08, max_value=0.08, right_digits=6)), 6),
        },
        'temperatura': round(float(fake.pyfloat(min_value=-5, max_value=45, right_digits=2)), 2),
        'humedad': round(float(fake.pyfloat(min_value=10, max_value=95, right_digits=2)), 2),
        'presion': round(float(fake.pyfloat(min_value=980, max_value=1065, right_digits=2)), 2),
        'co2': round(float(fake.pyfloat(min_value=350, max_value=2000, right_digits=2)), 2),
        'estado': fake.random_element(elements=estados),
        'timestamp': (start_ts + timedelta(minutes=i)).strftime('%Y-%m-%d %H:%M:%S')
    }

    records.append(record)
    i += 1

    if i % 500 == 0:
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False, separators=(',', ':'))

        if os.path.getsize(OUTPUT_FILE) >= TARGET_SIZE_BYTES:
            break

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, separators=(',', ':'))

real_size_bytes = os.path.getsize(OUTPUT_FILE)
real_size_mb = real_size_bytes / (1024 * 1024)

print(f'Registros generados: {len(records)}')
print(f'Fichero: {OUTPUT_FILE}')
print(f'Tamano real: {real_size_bytes} bytes ({real_size_mb:.2f} MB)')

In [12]:
# Ejemplo de primer registro
records[0]

{'sensor_id': 'sensor_01',
 'ubicacion': {'latitud': 43.331941, 'longitud': -3.162642},
 'temperatura': 2.86,
 'humedad': 79.11,
 'presion': 1055.54,
 'co2': 415.3,
 'estado': 'Activo',
 'timestamp': '2025-02-01 00:00:00'}

In [13]:
# Subida a S3 creando sesion explicita con credenciales temporales
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

s3 = session.client('s3')
s3.upload_file(OUTPUT_FILE, S3_BUCKET, S3_KEY)

print(f'Subida completada: s3://{S3_BUCKET}/{S3_KEY}')

Subida completada: s3://s3-streamlit-ra2/data/sensores/iabd05_sensores.json
